In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
import natural_units as nu
import scipy.integrate
from scipy.special import erf
from scipy.integrate import quad
from scipy.interpolate import RegularGridInterpolator
import re
# multi-core/thread:
import concurrent.futures

In [ ]:
speed_pdf_path = Path('../data/DM_Speed_PDF_finer')
eta_func_path = Path('../data/DM_eta_function_finer')
pattern = re.compile(r"mDM=([0-9.eE+-]+)_GeV_sigma=([0-9.eE+-]+)_cm2")

eta_func_path.mkdir(parents=True, exist_ok=True)
speed_pdf_jobs = {}
for filename in speed_pdf_path.glob('*.txt'):
    match = pattern.search(filename.stem)
    if match is None:
        continue
    key = (float(match.group(1)), float(match.group(2)))
    if key in speed_pdf_jobs:
        raise RuntimeError(f'Duplicate physical point in {speed_pdf_path}: {key}')
    output_path = eta_func_path / ('eta_func_' + filename.stem[13:] + '.txt')
    speed_pdf_jobs[key] = (filename, output_path)

if len(speed_pdf_jobs) != 49 * 17:
    raise RuntimeError(f'Expected 833 speed-PDF files, found {len(speed_pdf_jobs)}.')
speed_pdf_jobs = list(speed_pdf_jobs.values())
print(f'Selected {len(speed_pdf_jobs)} unique points on the 49 x 17 grid.')

In [3]:
def process_file(job):
    filename, output_path = job
    speed_pdf = np.loadtxt(filename, skiprows = 1)
    speed_recipro = speed_pdf[:,1] / speed_pdf[:,0]
    eta_func = np.zeros((speed_pdf.shape[0],2))
    eta_func[:,0] = speed_pdf[:,0]
    for i in range(speed_pdf.shape[0]):
        eta_func[i,1] = scipy.integrate.simpson(speed_recipro[i:], x = speed_pdf[i:,0])
    np.savetxt(output_path, eta_func)

In [4]:
def process_files_in_parallel(job_list):
    with concurrent.futures.ProcessPoolExecutor() as executor:
        list(executor.map(process_file, job_list))
process_files_in_parallel(speed_pdf_jobs)